In [1]:
import os, sys
from vertex_algebras.universal_vertex_algebra import UniversalVertexAlgebra
from vertex_algebras.normally_ordered_expressions import NormallyOrderedExpressions
from vertex_algebras.vertex_algebra import VertexAlgebraParallelism

In [2]:
R.<c>=QQ['c']
E.<L> = NormallyOrderedExpressions(R)
VertexAlgebraParallelism().set(27)   # threads of the C++ engine, for the whole session
MM= UniversalVertexAlgebra(R, E, {('L','L'): [L.derivative(),2*L, 0,(c/2)]}, conformal_vector=L)
L, = MM.gens()
print(L.ope(L))          # [T(L), 2*L, 0, c/2*1]
print(L*L)  # :L:L:

[dL, 2*L, 0, 1/2*c]
:L L:


In [3]:
def pole_matrix(a, k, source):
    """
    The matrix of  x |-> a.nproduct(x, k - 1)  (the coefficient of (z-w)^(-k) in a(z) x(w))
    from the weight space `source` to the weight space the images land in.
    Column j holds the coefficients of a.nproduct(b_j, k - 1) in the basis of the target space.
    Returns (matrix, target space).
    """
    # the images all live in one weight space: weight wt(a) + wt(source) - k, grade grade(a) + grade(source)
    W = a.degree() + source.weight() - k
    target = MM.weight_space(W)
    tkeys = target.basis_keys()                    # basis monomials of the target, as keys
    row = {m: r for r, m in enumerate(tkeys)}      # key -> row index
    basis = source.basis()
    M = matrix(MM.base_ring(), len(tkeys), len(basis))
    for j, b in enumerate(basis):
        image = a.nproduct(b, k - 1)
        # monomial_coefficients() is the dict {basis key: coefficient} of an element
        for m, c in image.monomial_coefficients().items():
            M[row[m], j] = c                       # a KeyError here would mean the image is not homogeneous
    return M, target

In [4]:
def solution_dimensions(A):
    """
    (generic dimension of {a : A a = 0}, {K0: dimension at K = K0} for the K0 where it jumps).
    """
    n = A.ncols()
    ed = A.elementary_divisors()
    nonzero = [d for d in ed if d != 0]
    special = {}
    for d in nonzero:
        if d.degree() > 0:
            for K0, _ in d.roots(QQbar):
                special[K0] = n - sum(1 for e in nonzero if e(K0) != 0)
    return n - len(nonzero), special

In [5]:
def solution_dimensions_fast(A, ncpus=27):
    """
    Same output as solution_dimensions(A), without the Smith form over QQ[c].

    The dimension of {a : A a = 0} can only jump at c0 where every N x N minor of A vanishes
    (N = generic rank), so the candidates are the roots of one such minor D(c).  D is found by
    evaluating the minor at deg(D)+1 rational points (independent determinants over QQ, run on
    `ncpus` processes) and interpolating; each candidate is then checked exactly by the rank of A(c0).
    """
    R = A.base_ring()
    n = A.ncols()
    # an N x N submatrix, N = generic rank, that is invertible at generic c
    # (the rank at a random point is the generic rank unless we hit a root)
    Ax = max((A.apply_map(lambda p: p(x)) for x in (QQ(1000003)/7, QQ(-999983)/11)), key=lambda M: M.rank())
    rows, cols = Ax.pivot_rows(), Ax.pivots()
    As = A.matrix_from_rows_and_columns(rows, cols)
    N = len(rows)
    dmax = max(e.degree() for e in A.list() if e != 0)
    pts = [QQ(i) for i in range(N * dmax + 1)]    # deg D <= N*dmax

    @parallel(ncpus=ncpus)
    def det_at(x):
        return As.apply_map(lambda p: p(x)).change_ring(QQ).det()

    vals = {args[0][0]: v for args, v in det_at(pts)}
    D = R.lagrange_polynomial([(x, vals[x]) for x in pts])

    special = {}
    for f, _ in D.factor():
        if f.degree() == 1:
            c0 = -f[0] / f[1]
            k = n - A.apply_map(lambda p: p(c0)).change_ring(QQ).rank()
            if k > n - N:
                special[c0] = k
        else:                                      # irrational roots: check once over the number field, conjugates agree
            K = NumberField(f, 'a')
            k = n - A.apply_map(lambda p: p(K.gen())).change_ring(K).rank()
            if k > n - N:
                for c0, _ in f.roots(QQbar):
                    special[c0] = k
    return n - N, special

In [6]:
def vectors_from(rows, source):
    """Coefficient vectors (w.r.t. source.basis()) -> elements of V."""
    basis = source.basis()
    return [sum(x * y for x, y in zip(row, basis)) for row in rows]

In [7]:
singular_vector_degree=22
conditions = [(L, 3), (L, 4)]          # (field a, pole order k):  a.nproduct(v, k - 1) = 0
A = block_matrix([[pole_matrix(a, k, MM.weight_space(singular_vector_degree))[0]] for a, k in conditions], subdivide=False)
print('A is', A.dimensions())
dims = solution_dimensions_fast(A, ncpus=7)   # solution_dimensions(A) gives the same, via the (slow, serial) Smith form
dims

A is (302, 210)


(0, {-1300/23: 1})

In [8]:
singular = {}
for c0 in dims[1]:
    c0 = QQ(c0)                                     # the roots are rational here
    A0 = A.apply_map(lambda p: p(c0))           # the matrix at c = c0, over QQ
    singular[c0] = vectors_from(A0.right_kernel_matrix().rows(), MM.weight_space(singular_vector_degree))
    for v in singular[c0]:
        print('c =', c0, ':', v, '\n')

c = -1300/23 : :L L L L L L L L L L L: - 165/23*:dL dL L L L L L L L L: + 4620/529*:dL dL dL dL L L L L L: - 15400/12167*:dL dL dL dL dL dL L L: - 495/46*:d^2L L L L L L L L L L: + 24486/529*:d^2L dL dL L L L L L L: - 281820/12167*:d^2L dL dL dL dL L L L: + 4620/12167*:d^2L dL dL dL dL dL dL: + 41811/1058*:d^2L d^2L L L L L L L L: - 2052105/24334*:d^2L d^2L dL dL L L L L: + 2729529/279841*:d^2L d^2L dL dL dL dL L: - 2870725/48668*:d^2L d^2L d^2L L L L L L: + 24237873/559682*:d^2L d^2L d^2L dL dL L L: + 147550821/4477456*:d^2L d^2L d^2L d^2L L L L: - 303448005/102981488*:d^2L d^2L d^2L d^2L dL dL: - 957151755/205962976*:d^2L d^2L d^2L d^2L d^2L L: + 44/529*:d^3L dL L L L L L L L: - 17380/12167*:d^3L dL dL dL L L L L: + 66440/279841*:d^3L dL dL dL dL dL L: - 30734/12167*:d^3L d^2L dL L L L L L: + 974732/279841*:d^3L d^2L dL dL dL L L: + 1631751/279841*:d^3L d^2L d^2L dL L L L: - 4310367/6436343*:d^3L d^2L d^2L dL dL dL: - 29073903/12872686*:d^3L d^2L d^2L d^2L dL L: - 550/1587*:d^3L d^3L

## The minimal model: the quotient by a singular vector

`singular` maps each special central charge `c0` found above to the singular vectors of weight `singular_vector_degree` in `MM`, the universal Virasoro vertex algebra over `QQ[c]`. Ideals and quotients need a base field and a fixed central charge, so below the universal Virasoro vertex algebra is built again over `QQ` at `c = c0`, the chosen singular vector is moved there (same basis monomials, coefficients evaluated at `c0`), and the quotient by the ideal it generates is formed.

At a minimal-model central charge $c_{p,p'} = 1 - 6(p-p')^2/(pp')$ the singular vector of weight $(p-1)(p'-1)$ generates the maximal ideal, so the quotient is the simple minimal model vertex algebra; its dimensions are compared with the vacuum character of $M(p,p')$ below.

Elements of the quotient are normal forms (canonical representatives modulo the ideal). They need the weight spaces of the ideal, which are computed on demand: the first products reaching weight `singular_vector_degree` and above take a while.

In [9]:
# the central charges with singular vectors (from the cells above)
for cc, vs in singular.items():
    print('c =', cc, ':', len(vs), 'singular vector(s) of weight', singular_vector_degree)

c0 = list(singular)[0]       # choose the central charge: one of the keys of `singular`
choice = 0                   # and the singular vector: an index into singular[c0]
v = singular[c0][choice]

c = -1300/23 : 1 singular vector(s) of weight 22


In [10]:
def virasoro_at(c0):
    '''
    The universal Virasoro vertex algebra over QQ at central charge c0, with L as its Virasoro vector
    (the OPEs are part of the algebra's data: another c0 gives another algebra).
    '''
    E = NormallyOrderedExpressions(QQ, ['L'])
    L0, = E.gens()
    return UniversalVertexAlgebra(QQ, E, {('L', 'L'): [L0.derivative(), 2*L0, 0, c0/2]},
                                  conformal_vector=L0)

def specialise(x, V, c0):
    '''The element x of MM (coefficients in QQ[c]) at c = c0, as an element of V (the same basis monomials).'''
    return V.sum_of_terms((m, QQ(p(c0))) for m, p in x.monomial_coefficients().items())

In [11]:
V = virasoro_at(c0)
LV, = V.gens()
v0 = specialise(v, V, c0)
# v0 is singular at c = c0: L_1 v0 = L_2 v0 = 0 (the conditions used above)
print('singular at c = %s:' % c0, all(LV.nproduct(v0, k - 1).is_zero() for k in (3, 4)))

I = V.ideal([v0], names=['v'])     # the ideal generated by v0
Q = V / I                          # the quotient vertex algebra, also V.quotient(I)
Q = Q.with_conformal_vector('ambient')   # the image of L is the Virasoro vector of the quotient
print('central charge of the quotient:', Q.central_charge())

singular at c = -1300/23: True
central charge of the quotient: -1300/23


In [12]:
%%time
W0 = singular_vector_degree
nmax = W0 + 4
dim_V = [V.weight_space(n).dimension() for n in range(nmax)]
dim_Q = [Q.weight_space(n).dimension() for n in range(nmax)]
print('weight    :', list(range(W0 - 2, nmax)))
print('universal :', dim_V[W0 - 2:])
print('quotient  :', dim_Q[W0 - 2:])

weight    : [20, 21, 22, 23, 24, 25]
universal : [137, 165, 210, 253, 320, 383]
quotient  : [137, 165, 209, 252, 318, 381]
CPU times: user 375 ms, sys: 50.8 ms, total: 425 ms
Wall time: 151 ms


In [13]:
# the minimal model M(p, p') with (p-1)(p'-1) = W0 and c_{p,p'} = c0, if there is one
pairs = [(p, W0 // (p - 1) + 1) for p in range(2, W0 + 2) if W0 % (p - 1) == 0]
pairs = [(p, pp) for p, pp in pairs if p < pp and gcd(p, pp) == 1 and 1 - 6*(p - pp)^2/(p*pp) == c0]
if pairs:
    p, pp = pairs[0]
    # vacuum character (Rocha-Caridi): sum_k ( q^(k(k p p' + p - p')) - q^((k p + 1)(k p' + 1)) ) / prod_n (1 - q^n)
    numer = [0] * nmax
    for k in range(-nmax, nmax + 1):
        for e, sign in ((k*(k*p*pp + p - pp), 1), ((k*p + 1)*(k*pp + 1), -1)):
            if 0 <= e < nmax:
                numer[e] += sign
    PS.<q> = PowerSeriesRing(ZZ, default_prec=nmax)
    chi = (PS(numer) / prod(1 - q^n for n in range(1, nmax))).padded_list(nmax)
    print('character of M(%d, %d):' % (p, pp), chi[W0 - 2:])
    print('the quotient is the simple minimal model up to weight %d:' % (nmax - 1), dim_Q == chi)
else:
    print("c0 is not a minimal-model central charge c_{p,p'} with (p-1)(p'-1) =", W0)

character of M(2, 23): [137, 165, 209, 252, 318, 381]
the quotient is the simple minimal model up to weight 25: True


In [14]:
# in the quotient the leading monomial of v0 is no longer independent: its image is a combination of
# standard monomials (those that are not leading monomials of the ideal)
lk = I.weight_space(W0).leading_monomials()[0]    # a basis key of V
lead = V.monomial(lk)
x = Q(lead)                        # the image of an element of V: its normal form
print('leading monomial of v0:', lead)
print('its image in the quotient is a combination of', len(x.support()), 'standard monomials, not including it:', lk not in x.support())
print('v0 maps to zero:', Q(v0).is_zero())
print('Q.lift(x) is its canonical representative in V:', Q.lift(x).parent() is V)

leading monomial of v0: :L L L L L L L L L L L:
its image in the quotient is a combination of 209 standard monomials, not including it: True
v0 maps to zero: True
Q.lift(x) is its canonical representative in V: True


Working in the quotient: `Q(y)` is the image of an element `y` of `V` (elements of `V` also coerce, e.g. `Q.gen('L') + y`), products of elements of `Q` (`.ope`, `.no`, `.nproduct`, `.derivative`) are computed in `V` and reduced to normal form, `Q.lift(x)` is the canonical representative, `Q.weight_space(n).basis()` lists the standard monomials, and `Q.ideal(...)`, `Q.subalgebra(...)`, `Q.tensor(...)` work as for any vertex algebra.